In [ ]:
import pandas as pd 
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 10000)
pd.set_option('display.width', 1000)


sheets = pd.ExcelFile(f'./PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx')
print(sheets.sheet_names)

data = pd.read_excel(f'./PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx', sheet_name='Table1 (basic)', skiprows=[0])
data.drop(['QualityRequirement', 'QualNo', 'CustID','CustVer','InternalVer','MetalRavneQualityName','SteelGroup','Month','Year'], axis=1, inplace=True)
data.drop_duplicates(subset=['HeatNo'],keep='first',inplace=True)

data.dropna(subset=['Cr_Last_EOP','Mo_Last_EOP','Ni_Last_EOP','V_Last_EOP','W_Last_EOP'],inplace=True)


data_=pd.read_excel('PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx',sheet_name='Table2 (limits)',skiprows=[0])
data_.drop(['QualityRequirement','QualNo','CustID','CustVer','InternalVer','MetalRavneQualityName','SteelGroup','Month','Year'],axis=1,inplace=True)
data_.drop(['Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final'],axis=1,inplace=True)
data_.drop_duplicates(subset=['HeatNo'], keep='first', inplace=True)

data=data.merge(data_,on='HeatNo',how='inner') 

data_=pd.read_excel('PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx',sheet_name='Table4 (alloys)',skiprows=[0])
data.drop(['LFVD_FeCrA', 'LFVD_FeCrC','LFVD_NiGran', 'LFVD_NiKatode','LFVD_FeMo', 'LFVD_Polymox','LFVD_FeV','LFVD_FeW72','LFVD_WPaketi'],axis=1,inplace=True)
data_.drop_duplicates(subset=['HeatNo'], keep='first', inplace=True)

data=data.merge(data_,on='HeatNo',how='inner')

# data_=pd.read_excel('PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx',sheet_name='Table7 (head)',skiprows=[0])
# data_.drop_duplicates(subset=['HeatNo'], keep='first', inplace=True)

# data=data.merge(data_,on='HeatNo',how='inner') # толоько для нахождения брака у работников

data_=pd.read_excel('PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx',sheet_name='Table8 (scrap)',skiprows=[0])
data_.drop_duplicates(subset=['HeatNo'], keep='first', inplace=True)

data=data.merge(data_,on='HeatNo',how='inner')

data_=pd.read_excel('PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx',sheet_name='Table9 (all. scr.)',skiprows=[0])
data_.drop_duplicates(subset=['HeatNo'], keep='first', inplace=True)

data=data.merge(data_,on='HeatNo',how='inner')

data_=pd.read_excel('PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx',sheet_name='Table10 (unall. scr.)',skiprows=[0])
data_.drop_duplicates(subset=['HeatNo'], keep='first', inplace=True)

data=data.merge(data_,on='HeatNo',how='inner')

display(data.head(3))
print('всего строк=',len(data))
print('всего столбцов=',len(data.columns))
data.to_csv('итоговая таблица.csv',sep=';',index=False)

### Удаление данных с нулевой диссперсией

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', 10000)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

import plotly.graph_objects as go
from plotly.subplots import make_subplots
data=pd.read_csv('итоговая таблица.csv',sep=';')
#LFVD_CaS LFVD_CoMet  LFVD_Cu LFVD_FeB	LFVD_FeBŽica	LFVD_FeCrNit  LFVD_FeMnA  LFVD_FeNbTa	LFVD_FeNbTafi13 LFVD_FeSiZrŽica	LFVD_FeTi	LFVD_FeTifi13
#LFVD_NiKatode	LFVD_NiOksid	LFVD_OdpCu  LFVD_SiMet  LFVD_KarboritZaVpih	LFVD_Ni90 

display(data.describe())
del_=[]
for i in data.describe().columns:
    if data.describe().loc['std',i]==0:
        del_.append(i)

del_.remove('Ni_LowerLimit')
del_.remove('Ni_Target')
del_.remove('W_LowerLimit')
del_.remove('W_Target')

data.drop(del_,axis=1,inplace=True)
data.drop(['Fill#1_ScrapName','Fill#1_ScrapWeight','Fill#2_ScrapName','Fill#2_ScrapWeight','Fill#3_ScrapName','Fill#3_ScrapWeight','Fill#4_ScrapName',
          'Fill#4_ScrapWeight','Fill#5_ScrapName','Fill#5_ScrapWeight','Fill#6_ScrapName','Fill#6_ScrapWeight'],axis=1,inplace=True)

print('всего строк=',len(data))
print('всего столбцов=',len(data.columns))
data.to_csv('итоговая таблица_испр.csv',sep=';',index=False)

### Проверка взаимных корреляций

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', 10000)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

import plotly.graph_objects as go
from plotly.subplots import make_subplots
data=pd.read_csv('итоговая таблица_испр.csv',sep=';')
data_=data.drop(['HeatNo','Date'],axis=1)

corr=data_.corr()

print('число столбцов в таблице=',len(data_.corr()))

display(corr.head(3))
gran_corr_value=0.8
corr_={i:[] for i in corr.columns}
for i in range(len(corr.columns)):    
    mas=corr[corr.columns[i]].tolist()
    el=mas[i]
    ind_max=[j for j in range(len(mas)) if (mas[j]>gran_corr_value)&(j!=i)]
    mas.remove(mas[i])
    corr_[corr.columns[i]]=[min(mas),max(mas)]+[corr.columns[j] for j in ind_max]
    print(corr.columns[i],':',corr_[corr.columns[i]])



### Гистограмма превышения целевого значения по компонентам

#### TODO
- создать гистограммы по отдельным сталеварам
- выявить случаи брака, превышение или занижение по элементу

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

data['Cr_exceed']=data.apply(lambda x:x['Cr_Final']-x['Cr_Target'],axis=1)
data['Ni_exceed']=data.apply(lambda x:x['Ni_Final']-x['Ni_Target'],axis=1)
data['Mo_exceed']=data.apply(lambda x:x['Mo_Final']-x['Mo_Target'],axis=1)
data['V_exceed']=data.apply(lambda x:x['V_Final']-x['V_Target'],axis=1)
data['W_exceed']=data.apply(lambda x:x['W_Final']-x['W_Target'],axis=1)

print('в среднем разница между Cr_Target и Cr_LowerLimit =',data.apply(lambda x:x['Cr_Target']-x['Cr_LowerLimit'],axis=1).mean())
print('в среднем разница между Ni_Target и Ni_LowerLimit =',data.apply(lambda x:x['Ni_Target']-x['Ni_LowerLimit'],axis=1).mean())
print('в среднем разница между Mo_Target и Mo_LowerLimit =',data.apply(lambda x:x['Mo_Target']-x['Mo_LowerLimit'],axis=1).mean())
print('в среднем разница между V_Target и V_LowerLimit =',data.apply(lambda x:x['V_Target']-x['V_LowerLimit'],axis=1).mean())
print('в среднем разница между W_Target и W_LowerLimit =',data.apply(lambda x:x['W_Target']-x['W_LowerLimit'],axis=1).mean())

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Histogram(x=data['Cr_exceed'],xbins=dict(start=-0.2, end= 0.2, size=0.005), name='Гистограмма превышения Cr'))
fig.add_trace(go.Histogram(x=data['Ni_exceed'],xbins=dict(start=-0.5, end= 0.5, size=0.005), name='Гистограмма превышения Ni'))
fig.add_trace(go.Histogram(x=data['Mo_exceed'],xbins=dict(start=-0.2, end= 0.2, size=0.005), name='Гистограмма превышения Mo'))
fig.add_trace(go.Histogram(x=data['V_exceed'],xbins=dict(start=-0.2, end= 0.2, size=0.005), name='Гистограмма превышения V'))
fig.add_trace(go.Histogram(x=data['W_exceed'],xbins=dict(start=-0.2, end= 0.2, size=0.005), name='Гистограмма превышения W'))
#fig.add_trace(go.Histogram(x=data['Cr_exceed'], name='Гистограмма превышения Cr'))
fig.show()

In [ ]:
data['Cr_adviser_target']=data.apply(lambda x:(x['Cr_Target']+x['Cr_LowerLimit'])/2,axis=1)
data['Ni_adviser_target']=data.apply(lambda x:(x['Ni_Target']+x['Ni_LowerLimit'])/2,axis=1)
data['Cr_exceed']=data.apply(lambda x:x['Cr_Final']-x['Cr_adviser_target'],axis=1)
data['Ni_exceed']=data.apply(lambda x:x['Ni_Final']-x['Ni_adviser_target'],axis=1)

print('в среднем разница между Cr_Target и Cr_LowerLimit =',data.apply(lambda x:x['Cr_Target']-x['Cr_LowerLimit'],axis=1).mean())
print('в среднем разница между Ni_Target и Ni_LowerLimit =',data.apply(lambda x:x['Ni_Target']-x['Ni_LowerLimit'],axis=1).mean())

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Histogram(x=data['Cr_exceed'],xbins=dict(start=-0.2, end= 0.2, size=0.005), name='Гистограмма превышения Cr'))
fig.add_trace(go.Histogram(x=data['Ni_exceed'],xbins=dict(start=-0.5, end= 0.5, size=0.005), name='Гистограмма превышения Ni'))
#fig.add_trace(go.Histogram(x=data['Cr_exceed'], name='Гистограмма превышения Cr'))
fig.show()

### Экономический эффект

In [ ]:
display(data[['TotalIngotsWeight','Cr_Last_EOP','Cr_Final','LFVD_FeCrA','LFVD_FeCrC',]].head(3))
print('количество строк с большим исходным составом=',len(data[(data['LFVD_FeCrA']==0)&(data['LFVD_FeCrC']==0)&(data['LFVD_FeCrCSi']==0)&(data['LFVD_FeCrC51']==0)]))

display(data[['TotalIngotsWeight','Ni_Last_EOP','Ni_Final','LFVD_NiGran']].head(3))
print('количество строк с большим исходным составом=',len(data[(data['LFVD_NiGran']==0)]))

display(data[['TotalIngotsWeight','Mo_Last_EOP','Mo_Final','LFVD_FeMo','LFVD_Polymox']].head(3))
print('количество строк с большим исходным составом=',len(data[(data['LFVD_FeMo']==0)&(data['LFVD_Polymox']==0)]))

display(data[['TotalIngotsWeight','V_Last_EOP','V_Final','LFVD_FeV']].head(3))
print('количество строк с большим исходным составом=',len(data[(data['LFVD_FeV']==0)]))

display(data[['TotalIngotsWeight','W_Last_EOP','W_Final','LFVD_FeW72']].head(3))
print('количество строк с большим исходным составом=',len(data[(data['LFVD_FeW72']==0)]))

In [ ]:
data_cr=data[(data['Cr_Final']>data['Cr_Last_EOP'])&(data['Cr_Final']>data['Cr_Target'])]
data_cr=data_cr.copy()

data_cr['FeCr_1%']=data_cr.apply(lambda x: (x['LFVD_FeCrA']+x['LFVD_FeCrC'])/(x['Cr_Final']-x['Cr_Last_EOP'])/x['TotalIngotsWeight']*100,axis=1)
data_cr['Cr_surplus']=data_cr.apply(lambda x:(x['LFVD_FeCrA']+x['LFVD_FeCrC'])-(x['Cr_Target']-x['Cr_Last_EOP'])*x['TotalIngotsWeight']*x['FeCr_1%']/100,axis=1)
display(data_cr[['TotalIngotsWeight','Cr_Last_EOP','Cr_Target','Cr_Final','LFVD_FeCrA','LFVD_FeCrC','FeCr_1%','Cr_surplus']].head(3))
print('перерасход в год FeCr=',round(data_cr['Cr_surplus'].sum()/10,0),'кг',' примерная стоимость=',round(data_cr['Cr_surplus'].sum()/10*300/1000000,2),'млн.руб')

In [ ]:
data_mo=data[(data['Mo_Final']>data['Mo_Last_EOP'])&(data['Mo_Final']>data['Mo_Target'])]
data_mo=data_mo.copy()

data_mo['FeMo_1%']=data_mo.apply(lambda x: (x['LFVD_FeMo']+x['LFVD_Polymox'])/(x['Mo_Final']-x['Mo_Last_EOP'])/x['TotalIngotsWeight']*100,axis=1)
data_mo['Mo_surplus']=data_mo.apply(lambda x:(x['LFVD_FeMo']+x['LFVD_Polymox'])-(x['Mo_Target']-x['Mo_Last_EOP'])*x['TotalIngotsWeight']*x['FeMo_1%']/100,axis=1)
display(data_mo[['TotalIngotsWeight','Mo_Last_EOP','Mo_Target','Mo_Final','LFVD_FeMo','FeMo_1%','Mo_surplus']].head(3))
print('перерасход в год Mo=',round(data_mo['Mo_surplus'].sum()/10,0),'кг',' примерная стоимость=',round(data_mo['Mo_surplus'].sum()/10*1500/1000000,2),'млн.руб')

In [ ]:
data_v=data[(data['V_Final']>data['V_Last_EOP'])&(data['V_Final']>data['V_Target'])]
data_v=data_v.copy()

data_v['FeV_1%']=data_v.apply(lambda x: (x['LFVD_FeV'])/(x['V_Final']-x['V_Last_EOP'])/x['TotalIngotsWeight']*100,axis=1)
data_v['V_surplus']=data_v.apply(lambda x:(x['LFVD_FeV'])-(x['V_Target']-x['V_Last_EOP'])*x['TotalIngotsWeight']*x['FeV_1%']/100,axis=1)
display(data_v[['TotalIngotsWeight','V_Last_EOP','V_Target','V_Final','LFVD_FeV','FeV_1%','V_surplus']].head(3))
print('перерасход в год V=',round(data_v['V_surplus'].sum()/10,0),'кг',' примерная стоимость=',round(data_v['V_surplus'].sum()/10*1500/1000000,2),'млн.руб')

In [ ]:
data_w=data[(data['W_Final']>data['W_Last_EOP'])&(data['W_Final']>data['W_Target'])]
data_w=data_w.copy()

data_w['FeW_1%']=data_w.apply(lambda x: (x['LFVD_FeW72'])/(x['W_Final']-x['W_Last_EOP'])/x['TotalIngotsWeight']*100,axis=1)
data_w['W_surplus']=data_w.apply(lambda x:(x['LFVD_FeW72'])-(x['W_Target']-x['W_Last_EOP'])*x['TotalIngotsWeight']*x['FeW_1%']/100,axis=1)
display(data_w[['TotalIngotsWeight','W_Last_EOP','W_Target','W_Final','LFVD_FeW72','FeW_1%','W_surplus']].head(3))
print('перерасход в год W=',round(data_w['W_surplus'].sum()/10,0),'кг',' примерная стоимость=',round(data_w['W_surplus'].sum()/10*1500/1000000,2),'млн.руб')

### Данные по отдельным сталеварам

In [ ]:
#'Cr_surplus', 'Mo_surplus', 'V_surplus', 'W_surplus', 'Total_defects' 
data_unique_emp = pd.DataFrame(data['LFVD_ShiftHead'].unique(), columns=['LFVD_ShiftHead'])

fig = make_subplots(specs=[[{"secondary_y": True}]])

for emp in data_unique_emp['LFVD_ShiftHead']:
    deffects = data.agg(lambda x: 1 if x['LFVD_ShiftHead']==emp and (x['Cr_Final'] > x['Cr_Last_EOP']) and (x['Cr_Final'] > x['Cr_Target']) else 0, axis=1).sum()
    data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'Cr_defects']=deffects 
    data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'Total_defects']=deffects
    deffects = data.agg(lambda x: 1 if x['LFVD_ShiftHead']==emp and (x['Ni_Final'] > x['Ni_Last_EOP']) and (x['Ni_Final'] > x['Ni_Target']) else 0, axis=1).sum()
    data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'Ni_defects']=deffects 
    data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'Total_defects']+=deffects
    deffects = data.agg(lambda x: 1 if x['LFVD_ShiftHead']==emp and (x['Mo_Final'] > x['Mo_Last_EOP']) and (x['Mo_Final'] > x['Mo_Target']) else 0, axis=1).sum()
    data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'Mo_defects']=deffects 
    data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'Total_defects']+=deffects
    deffects = data.agg(lambda x: 1 if x['LFVD_ShiftHead']==emp and (x['V_Final'] > x['V_Last_EOP']) and (x['V_Final'] > x['V_Target']) else 0, axis=1).sum()
    data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'V_defects']=deffects 
    data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'Total_defects']+=deffects
    deffects = data.agg(lambda x: 1 if x['LFVD_ShiftHead']==emp and (x['W_Final'] > x['W_Last_EOP']) and (x['W_Final'] > x['W_Target']) else 0, axis=1).sum()
    data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'W_defects']=deffects
    data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'Total_defects']+=deffects

for emp in data_unique_emp:
    fig.add_trace(go.Bar(x=[emp], y=[data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'Total_defects']], name=f'Гистограмма дефектов для {emp}'))
    

fig.show()
display(data_unique_emp)



### Какая стоимость перерасхода если нам удастся обеспечить среднее значение между Cr_LowerLimit и Cr_Target?

### Выполните расчеты для каждого элемента: Mo, Ni, V, W

In [ ]:
import pandas as pd
import dtreeviz
from sklearn.tree import DecisionTreeClassifier


random_state=42
data=pd.read_csv('итоговая таблица_испр.csv',sep=';')
data.fillna(0,inplace=True)
#display(data)
data['quality']=data.apply(lambda x: 1 if x['Cr_Final']<=x['Cr_Target'] else 0,axis=1)

X=data.drop(['HeatNo','quality','Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final'],axis=1).values
y=data['quality'].values

features=data.drop(['HeatNo','Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final'],axis=1).columns.tolist()
class_names=list(set(y))
target='quality'
tree_classifier = DecisionTreeClassifier(max_depth=33, random_state=random_state)
tree_classifier.fit(X, y)

# viz_model = dtreeviz.model(tree_classifier,
#                            X_train=X, y_train=y,
#                            feature_names=features,
#                            target_name=target)

#viz_model.view(scale=0.8)
#viz_model.save('dtreeviz.svg')

In [3]:
import pandas as pd
import dtreeviz
from sklearn.tree import DecisionTreeRegressor
import dtreeviz

pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

random_state=42
data=pd.read_csv('итоговая таблица_испр.csv',sep=';')
data.fillna(0,inplace=True)
trees = []
#display(data)
qual = ['Cr_Final', 'Ni_Final','Mo_Final','V_Final', 'W_Final']
for i in range(5):
    X=data.drop(['HeatNo','Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final'],axis=1).values #['HeatNo','quality','Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final']
    y=data[qual[i]].values
    features=data.drop(['HeatNo','Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final'],axis=1).columns.tolist()  #['HeatNo','quality','Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final']
    tree_reggression = DecisionTreeRegressor(max_depth=33, random_state=random_state)
    trees.append(tree_reggression.fit(X, y))
trees = dict(zip(qual, trees))
print(trees)
print('Деревья обучены')

{'Cr_Final': DecisionTreeRegressor(max_depth=33, random_state=42), 'Ni_Final': DecisionTreeRegressor(max_depth=33, random_state=42), 'Mo_Final': DecisionTreeRegressor(max_depth=33, random_state=42), 'V_Final': DecisionTreeRegressor(max_depth=33, random_state=42), 'W_Final': DecisionTreeRegressor(max_depth=33, random_state=42)}
Деревья обучены


### Метрики точности моделей
Здесь используем оценки:
1. mE = (1/n)sum(y`i - yi)
2. MAE or MSE or RMSE
3. R correlation

In [65]:
data=pd.read_csv('итоговая таблица_испр.csv',sep=';')
data.fillna(0,inplace=True)
check = data.drop(['HeatNo','Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final'],axis=1) #.sample(frac=0.2)
ans = pd.DataFrame()
for i in range(0, len(check)):
    line = pd.DataFrame()
    for j in trees:
        line[j] = trees[j].predict(check.iloc[i].values.reshape(1, -1))
    ans = pd.concat([ans, line], ignore_index=True)
display(ans)

evaluations = pd.DataFrame(columns=['Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final'], index=['mE', 'MSE', 'R_correlation'])
s = 0
for j in evaluations.columns:
    for i in range(0, len(data)):
        s += data.iloc[i].loc[j] - ans.iloc[i].loc[j]
    evaluations.loc['mE', j] = s / len(data)

for j in evaluations.columns:
    for i in range(0, len(data)):
        s += (data.iloc[i].loc[j] - ans.iloc[i].loc[j])**2
    evaluations.loc['MSE', j] = s / len(data)

for j in evaluations.columns:
    evaluations.loc['R_correlation', j] = data[j].corr(ans[j])
    
display(evaluations)

,Cr_Final,Ni_Final,Mo_Final,V_Final,W_Final
0,11.30,0.21,0.71,0.90,0.02
1,11.75,0.16,0.85,0.29,0.02
2,11.63,0.18,0.72,0.91,0.02
3,11.60,0.17,0.73,0.92,0.01
4,11.29,0.21,0.73,0.74,0.12
...,...,...,...,...,...
3220,11.42,0.14,0.75,0.75,0.01
3221,11.30,0.13,0.76,0.77,0.04
3222,12.07,0.15,0.99,0.75,0.02
3223,11.29,0.16,0.72,0.77,0.03


,Cr_Final,Ni_Final,Mo_Final,V_Final,W_Final
mE,0.0,0.0,-0.0,-0.0,-0.0
MSE,0.0,0.0,0.000001,0.000005,0.000005
R_correlation,0.999996,1.0,0.999864,0.999807,1.0


In [ ]:
import pandas as pd
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

import random as rnd

data = pd.read_csv('итоговая таблица_испр.csv', sep=';')

# example_data = data.iloc[rnd.randint(0, len(data))].loc[['PouringScrap', 'OtherScrap','TotalIngotsWeight',
#                                                          'Cr_Last_EOP', 'Cr_LowerLimit', 'Cr_UpperLimit', 'Cr_Target', 
#                                                          'Ni_Last_EOP', 'Ni_LowerLimit', 'Ni_UpperLimit', 'Ni_Target',
#                                                          'Mo_Last_EOP', 'Mo_LowerLimit', 'Mo_UpperLimit', 'Mo_Target',
#                                                          'V_Last_EOP', 'V_LowerLimit', 'V_UpperLimit', 'V_Target',
#                                                          'W_Last_EOP', 'W_LowerLimit', 'W_UpperLimit', 'W_Target']].to_frame() #data from historical
example_data = pd.DataFrame(data={'PouringScrap':[300], 'OtherScrap':[200], 'TotalIngotsWeight': [32300],
                                   'Cr_Last_EOP':[10.2019], 'Cr_LowerLimit':[11.1], 'Cr_UpperLimit':[11.75], 'Cr_Target':[11.46],
                                   'Ni_Last_EOP':[0.1], 'Ni_LowerLimit':[0], 'Ni_UpperLimit': [0.3], 'Ni_Target':[0],
                                    'Mo_Last_EOP':[0.8], 'Mo_LowerLimit':[0.7], 'Mo_UpperLimit':[0.85], 'Mo_Target':[0.73],
                                    'V_Last_EOP':[0.73], 'V_LowerLimit':[0.72], 'V_UpperLimit':[0.85], 'V_Target':[0.76],
                                    'W_Last_EOP':[0.4], 'W_LowerLimit':[0], 'W_UpperLimit':[0.2], 'W_Target':[0]}).T #random data
other_par = data.drop(['Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final'],axis=1)
ans = pd.DataFrame(columns=other_par.columns.to_list())
#display(ans)

f = [['Cr_LowerLimit', 'Cr_Final', 'Cr_UpperLimit'],
     ['Ni_LowerLimit', 'Ni_Final', 'Ni_UpperLimit'],
     ['Mo_LowerLimit', 'Mo_Final', 'Mo_UpperLimit'],
     ['V_LowerLimit', 'V_Final', 'V_UpperLimit'],
     ['W_LowerLimit', 'W_Final', 'W_UpperLimit']]
ans = dict.fromkeys(['Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final'], pd.DataFrame())

for j in f:
    for i in range(0, len(other_par)):
        miss = other_par.columns.difference(example_data.T.columns).to_list()
        data_ = example_data.T

        # display(len(miss))
        # display(data_)
        # display(other_par.loc[i, miss].to_frame().T)

        # other_par.loc[i, miss].to_frame().T
        # print(other_par)
        data_[miss] = other_par.loc[i, miss].to_list()
        #print(data_)
        
        # display(len(data_.columns))
        if example_data.loc[j[0]].values[0] < trees[j[1]].predict(data_.loc[:, data_.columns != 'HeatNo'].values) and trees[j[1]].predict(data_.loc[:, data_.columns != 'HeatNo'].values) < example_data.loc[j[2]].values[0]: #example_data.loc[j[0]].values[0] < trees[j[1]].predict(data_.loc[:, data_.columns != 'HeatNo'].values) and 
            ans[j[1]] = pd.concat([ans[j[1]], data_])
            
    print(len(ans[j[1]]))  
        
    
for j in ans.keys():
    ans[j].reset_index()
    display(ans[j])


3024
1
0
0
3225


,PouringScrap,OtherScrap,TotalIngotsWeight,Cr_Last_EOP,Cr_LowerLimit,Cr_UpperLimit,Cr_Target,Ni_Last_EOP,Ni_LowerLimit,Ni_UpperLimit,Ni_Target,Mo_Last_EOP,Mo_LowerLimit,Mo_UpperLimit,Mo_Target,V_Last_EOP,V_LowerLimit,V_UpperLimit,V_Target,W_Last_EOP,W_LowerLimit,W_UpperLimit,W_Target,HeatNo,LFVD_AlBloki,LFVD_AlGran,LFVD_AlOplašèenaŽica,LFVD_AlŽica,LFVD_BelaŽlindra,LFVD_Boksit,LFVD_CASIfi13,LFVD_CaO,LFVD_Cfi13,LFVD_EPŽŽlindra,LFVD_FeAl,LFVD_FeCrA,LFVD_FeCrC,LFVD_FeCrC51,LFVD_FeCrCSi,LFVD_FeMnC,LFVD_FeMo,LFVD_FeS,LFVD_FeSi,LFVD_FeV,LFVD_FeW72,LFVD_KalcijevKarbid,LFVD_KarboritMleti,LFVD_MnMet,LFVD_NiGran,LFVD_Polymox,LFVD_SLAGMAG65B,LFVD_SiMn,LFVD_SintŽlindra,LFVD_SŽica,Last_EOP,PV_31CRV3,PV_CRMO_OSTRUŽKI,PV_CRV,PV_E3,PV_E6,PV_E8,PV_EMCR,PV_OCR12,PV_OCR12SP,PV_OCR12VM,PV_OCR12VM_OSTRUŽKI,PV_OH252,PV_OH255,PV_PK2,PV_PK5,PV_PK5M,PV_PT181,PV_UTOPMO2,PV_UTOPMO4,PV_VCMO230
0,300.0,200.0,32300.0,10.2019,11.1,11.75,11.46,0.1,0.0,0.3,0.0,0.8,0.7,0.85,0.73,0.73,0.72,0.85,0.76,0.4,0.0,0.2,0.0,73590.0,20.0,0.0,0.0,0.0,0.0,100.0,16.5,400.0,0.0,0.0,0.0,0.0,1250.0,0.0,0.0,0.0,235.0,0.0,40.0,280.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,15200.0,0.0,0.0,0.0,0.0,25000.0,5000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0,300.0,200.0,32300.0,10.2019,11.1,11.75,11.46,0.1,0.0,0.3,0.0,0.8,0.7,0.85,0.73,0.73,0.72,0.85,0.76,0.4,0.0,0.2,0.0,73597.0,16.0,0.0,0.0,0.0,0.0,150.0,16.5,400.0,0.0,0.0,0.0,0.0,700.0,0.0,0.0,30.0,235.0,0.0,100.0,170.0,0.0,0.0,60.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,15100.0,0.0,0.0,0.0,0.0,23000.0,5000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0,300.0,200.0,32300.0,10.2019,11.1,11.75,11.46,0.1,0.0,0.3,0.0,0.8,0.7,0.85,0.73,0.73,0.72,0.85,0.76,0.4,0.0,0.2,0.0,73598.0,19.0,0.0,0.0,0.0,0.0,150.0,16.5,450.0,0.0,0.0,0.0,200.0,700.0,0.0,0.0,70.0,195.0,0.0,50.0,240.0,0.0,0.0,25.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,15200.0,0.0,0.0,0.0,0.0,25000.0,5000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0,300.0,200.0,32300.0,10.2019,11.1,11.75,11.46,0.1,0.0,0.3,0.0,0.8,0.7,0.85,0.73,0.73,0.72,0.85,0.76,0.4,0.0,0.2,0.0,73599.0,20.0,0.0,0.0,0.0,0.0,170.0,16.5,300.0,0.0,0.0,0.0,0.0,1300.0,0.0,0.0,0.0,250.0,0.0,80.0,275.0,0.0,0.0,25.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,15200.0,0.0,0.0,0.0,0.0,25000.0,5000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0,300.0,200.0,32300.0,10.2019,11.1,11.75,11.46,0.1,0.0,0.3,0.0,0.8,0.7,0.85,0.73,0.73,0.72,0.85,0.76,0.4,0.0,0.2,0.0,73600.0,5.0,0.0,0.0,2.9,0.0,60.0,16.5,250.0,12.0,0.0,0.0,0.0,1200.0,0.0,0.0,0.0,220.0,0.0,90.0,235.0,0.0,0.0,30.0,0.0,0.0,0.0,0.0,85.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,15500.0,0.0,0.0,0.0,0.0,25000.0,5000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,300.0,200.0,32300.0,10.2019,11.1,11.75,11.46,0.1,0.0,0.3,0.0,0.8,0.7,0.85,0.73,0.73,0.72,0.85,0.76,0.4,0.0,0.2,0.0,98222.0,5.0,10.0,0.0,0.0,0.0,160.0,14.3,450.0,5.5,0.0,0.0,0.0,290.0,0.0,0.0,40.0,175.0,0.0,30.0,266.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,24000.0,0.0,0.0,0.0,0.0,15000.0,5000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0,300.0,200.0,32300.0,10.2019,11.1,11.75,11.46,0.1,0.0,0.3,0.0,0.8,0.7,0.85,0.73,0.73,0.72,0.85,0.76,0.4,0.0,0.2,0.0,98223.0,5.0,10.0,0.0,0.0,0.0,310.0,15.2,600.0,0.0,0.0,0.0,500.0,16.0,0.0,0.0,0.0,51.0,0.0,0.0,337.0,0.0,0.0,0.0,60.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,24500.0,0.0,0.0,0.0,0.0,15000.0,5000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0,300.0,200.0,32300.0,10.2019,11.1,11.75,11.46,0.1,0.0,0.3,0.0,0.8,0.7,0.85,0.73,0.73,0.72,0.85,0.76,0.4,0.0,0.2,0.0,98244.0,13.0,0.0,0.0,0.0,0.0,480.0,14.3,850.0,7.4,0.0,0.0,0.0,1960.0,0.0,0.0,128.0,113.0,0.0,100.0,328.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,40.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,22000.0,0.0,0.0,0.0,0.0,15000.0,5000.0,0.0,0.0,0

,PouringScrap,OtherScrap,TotalIngotsWeight,Cr_Last_EOP,Cr_LowerLimit,Cr_UpperLimit,Cr_Target,Ni_Last_EOP,Ni_LowerLimit,Ni_UpperLimit,Ni_Target,Mo_Last_EOP,Mo_LowerLimit,Mo_UpperLimit,Mo_Target,V_Last_EOP,V_LowerLimit,V_UpperLimit,V_Target,W_Last_EOP,W_LowerLimit,W_UpperLimit,W_Target,HeatNo,LFVD_AlBloki,LFVD_AlGran,LFVD_AlOplašèenaŽica,LFVD_AlŽica,LFVD_BelaŽlindra,LFVD_Boksit,LFVD_CASIfi13,LFVD_CaO,LFVD_Cfi13,LFVD_EPŽŽlindra,LFVD_FeAl,LFVD_FeCrA,LFVD_FeCrC,LFVD_FeCrC51,LFVD_FeCrCSi,LFVD_FeMnC,LFVD_FeMo,LFVD_FeS,LFVD_FeSi,LFVD_FeV,LFVD_FeW72,LFVD_KalcijevKarbid,LFVD_KarboritMleti,LFVD_MnMet,LFVD_NiGran,LFVD_Polymox,LFVD_SLAGMAG65B,LFVD_SiMn,LFVD_SintŽlindra,LFVD_SŽica,Last_EOP,PV_31CRV3,PV_CRMO_OSTRUŽKI,PV_CRV,PV_E3,PV_E6,PV_E8,PV_EMCR,PV_OCR12,PV_OCR12SP,PV_OCR12VM,PV_OCR12VM_OSTRUŽKI,PV_OH252,PV_OH255,PV_PK2,PV_PK5,PV_PK5M,PV_PT181,PV_UTOPMO2,PV_UTOPMO4,PV_VCMO230
0,300.0,200.0,32300.0,10.2019,11.1,11.75,11.46,0.1,0.0,0.3,0.0,0.8,0.7,0.85,0.73,0.73,0.72,0.85,0.76,0.4,0.0,0.2,0.0,75290.0,20.0,0.0,3.0,0.0,0.0,70.0,16.5,400.0,17.2,0.0,0.0,0.0,1500.0,0.0,0.0,30.0,235.0,0.0,120.0,630.0,0.0,0.0,135.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,26400.0,0.0,0.0,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


""


""


,PouringScrap,OtherScrap,TotalIngotsWeight,Cr_Last_EOP,Cr_LowerLimit,Cr_UpperLimit,Cr_Target,Ni_Last_EOP,Ni_LowerLimit,Ni_UpperLimit,Ni_Target,Mo_Last_EOP,Mo_LowerLimit,Mo_UpperLimit,Mo_Target,V_Last_EOP,V_LowerLimit,V_UpperLimit,V_Target,W_Last_EOP,W_LowerLimit,W_UpperLimit,W_Target,HeatNo,LFVD_AlBloki,LFVD_AlGran,LFVD_AlOplašèenaŽica,LFVD_AlŽica,LFVD_BelaŽlindra,LFVD_Boksit,LFVD_CASIfi13,LFVD_CaO,LFVD_Cfi13,LFVD_EPŽŽlindra,LFVD_FeAl,LFVD_FeCrA,LFVD_FeCrC,LFVD_FeCrC51,LFVD_FeCrCSi,LFVD_FeMnC,LFVD_FeMo,LFVD_FeS,LFVD_FeSi,LFVD_FeV,LFVD_FeW72,LFVD_KalcijevKarbid,LFVD_KarboritMleti,LFVD_MnMet,LFVD_NiGran,LFVD_Polymox,LFVD_SLAGMAG65B,LFVD_SiMn,LFVD_SintŽlindra,LFVD_SŽica,Last_EOP,PV_31CRV3,PV_CRMO_OSTRUŽKI,PV_CRV,PV_E3,PV_E6,PV_E8,PV_EMCR,PV_OCR12,PV_OCR12SP,PV_OCR12VM,PV_OCR12VM_OSTRUŽKI,PV_OH252,PV_OH255,PV_PK2,PV_PK5,PV_PK5M,PV_PT181,PV_UTOPMO2,PV_UTOPMO4,PV_VCMO230
0,300.0,200.0,32300.0,10.2019,11.1,11.75,11.46,0.1,0.0,0.3,0.0,0.8,0.7,0.85,0.73,0.73,0.72,0.85,0.76,0.4,0.0,0.2,0.0,73579.0,26.0,0.0,0.0,0.0,0.0,200.0,16.5,600.0,20.6,0.0,0.0,0.0,2400.0,0.0,0.0,20.0,520.0,0.0,130.0,580.0,0.0,0.0,125.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,40000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0,300.0,200.0,32300.0,10.2019,11.1,11.75,11.46,0.1,0.0,0.3,0.0,0.8,0.7,0.85,0.73,0.73,0.72,0.85,0.76,0.4,0.0,0.2,0.0,73580.0,20.0,0.0,0.0,0.0,0.0,100.0,16.5,500.0,0.0,0.0,0.0,0.0,1200.0,0.0,0.0,100.0,625.0,0.0,55.0,135.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,40200.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0,300.0,200.0,32300.0,10.2019,11.1,11.75,11.46,0.1,0.0,0.3,0.0,0.8,0.7,0.85,0.73,0.73,0.72,0.85,0.76,0.4,0.0,0.2,0.0,73581.0,15.0,0.0,0.0,0.0,0.0,70.0,16.5,400.0,0.0,0.0,0.0,0.0,2000.0,0.0,0.0,0.0,490.0,0.0,80.0,520.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,40200.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0,300.0,200.0,32300.0,10.2019,11.1,11.75,11.46,0.1,0.0,0.3,0.0,0.8,0.7,0.85,0.73,0.73,0.72,0.85,0.76,0.4,0.0,0.2,0.0,73589.0,20.0,0.0,0.0,0.0,0.0,200.0,16.5,400.0,20.6,0.0,0.0,0.0,1450.0,0.0,0.0,60.0,515.0,0.0,80.0,565.0,0.0,0.0,60.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,40200.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0,300.0,200.0,32300.0,10.2019,11.1,11.75,11.46,0.1,0.0,0.3,0.0,0.8,0.7,0.85,0.73,0.73,0.72,0.85,0.76,0.4,0.0,0.2,0.0,73590.0,20.0,0.0,0.0,0.0,0.0,100.0,16.5,400.0,0.0,0.0,0.0,0.0,1250.0,0.0,0.0,0.0,235.0,0.0,40.0,280.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,15200.0,0.0,0.0,0.0,0.0,25000.0,5000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,300.0,200.0,32300.0,10.2019,11.1,11.75,11.46,0.1,0.0,0.3,0.0,0.8,0.7,0.85,0.73,0.73,0.72,0.85,0.76,0.4,0.0,0.2,0.0,98222.0,5.0,10.0,0.0,0.0,0.0,160.0,14.3,450.0,5.5,0.0,0.0,0.0,290.0,0.0,0.0,40.0,175.0,0.0,30.0,266.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,24000.0,0.0,0.0,0.0,0.0,15000.0,5000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0,300.0,200.0,32300.0,10.2019,11.1,11.75,11.46,0.1,0.0,0.3,0.0,0.8,0.7,0.85,0.73,0.73,0.72,0.85,0.76,0.4,0.0,0.2,0.0,98223.0,5.0,10.0,0.0,0.0,0.0,310.0,15.2,600.0,0.0,0.0,0.0,500.0,16.0,0.0,0.0,0.0,51.0,0.0,0.0,337.0,0.0,0.0,0.0,60.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,24500.0,0.0,0.0,0.0,0.0,15000.0,5000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0,300.0,200.0,32300.0,10.2019,11.1,11.75,11.46,0.1,0.0,0.3,0.0,0.8,0.7,0.85,0.73,0.73,0.72,0.85,0.76,0.4,0.0,0.2,0.0,98244.0,13.0,0.0,0.0,0.0,0.0,480.0,14.3,850.0,7.4,0.0,0.0,0.0,1960.0,0.0,0.0,128.0,113.0,0.0,100.0,328.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,40.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,22000.0,0.0,0.0,0.0,0.0,15000.0,5000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.

In [59]:
#/////////////// UNDER CONSTRUCTION BLOCK ///////////////////
last = ans['Cr_Final']
# last = last.merge(ans['Ni_Final'], on='HeatNo', how='inner', suffixes=['Cr','Ni'])
for i in ['Ni_Final','Mo_Final','V_Final','W_Final']: #columns set by priority 
    try:
        last = last.merge(ans[i], on='HeatNo', how='inner', suffixes=['_a', '_b'])
    except:
        print('Error' + i)
        continue
last = last.reset_index()
display(last)

ErrorMo_Final
ErrorV_Final


C:\Users\Amega\AppData\Local\Temp\ipykernel_18448\202609980.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


,index,PouringScrap_a,OtherScrap_a,TotalIngotsWeight_a,Cr_Last_EOP_a,Cr_LowerLimit_a,Cr_UpperLimit_a,Cr_Target_a,Ni_Last_EOP_a,Ni_LowerLimit_a,Ni_UpperLimit_a,Ni_Target_a,Mo_Last_EOP_a,Mo_LowerLimit_a,Mo_UpperLimit_a,Mo_Target_a,V_Last_EOP_a,V_LowerLimit_a,V_UpperLimit_a,V_Target_a,W_Last_EOP_a,W_LowerLimit_a,W_UpperLimit_a,W_Target_a,HeatNo,LFVD_AlBloki_a,LFVD_AlGran_a,LFVD_AlOplašèenaŽica_a,LFVD_AlŽica_a,LFVD_BelaŽlindra_a,LFVD_Boksit_a,LFVD_CASIfi13_a,LFVD_CaO_a,LFVD_Cfi13_a,LFVD_EPŽŽlindra_a,LFVD_FeAl_a,LFVD_FeCrA_a,LFVD_FeCrC_a,LFVD_FeCrC51_a,LFVD_FeCrCSi_a,LFVD_FeMnC_a,LFVD_FeMo_a,LFVD_FeS_a,LFVD_FeSi_a,LFVD_FeV_a,LFVD_FeW72_a,LFVD_KalcijevKarbid_a,LFVD_KarboritMleti_a,LFVD_MnMet_a,LFVD_NiGran_a,LFVD_Polymox_a,LFVD_SLAGMAG65B_a,LFVD_SiMn_a,LFVD_SintŽlindra_a,LFVD_SŽica_a,Last_EOP_a,PV_31CRV3_a,PV_CRMO_OSTRUŽKI_a,PV_CRV_a,PV_E3_a,PV_E6_a,PV_E8_a,PV_EMCR_a,PV_OCR12_a,PV_OCR12SP_a,PV_OCR12VM_a,PV_OCR12VM_OSTRUŽKI_a,PV_OH252_a,PV_OH255_a,PV_PK2_a,PV_PK5_a,PV_PK5M_a,PV_PT181_a,PV_UTOPMO2_a,PV_UTOPMO4_a,PV_VCMO230_a,PouringScrap_b,OtherScrap_b,TotalIngotsWeight_b,Cr_Last_EOP_b,Cr_LowerLimit_b,Cr_UpperLimit_b,Cr_Target_b,Ni_Last_EOP_b,Ni_LowerLimit_b,Ni_UpperLimit_b,Ni_Target_b,Mo_Last_EOP_b,Mo_LowerLimit_b,Mo_UpperLimit_b,Mo_Target_b,V_Last_EOP_b,V_LowerLimit_b,V_UpperLimit_b,V_Target_b,W_Last_EOP_b,W_LowerLimit_b,W_UpperLimit_b,W_Target_b,LFVD_AlBloki_b,LFVD_AlGran_b,LFVD_AlOplašèenaŽica_b,LFVD_AlŽica_b,LFVD_BelaŽlindra_b,LFVD_Boksit_b,LFVD_CASIfi13_b,LFVD_CaO_b,LFVD_Cfi13_b,LFVD_EPŽŽlindra_b,LFVD_FeAl_b,LFVD_FeCrA_b,LFVD_FeCrC_b,LFVD_FeCrC51_b,LFVD_FeCrCSi_b,LFVD_FeMnC_b,LFVD_FeMo_b,LFVD_FeS_b,LFVD_FeSi_b,LFVD_FeV_b,LFVD_FeW72_b,LFVD_KalcijevKarbid_b,LFVD_KarboritMleti_b,LFVD_MnMet_b,LFVD_NiGran_b,LFVD_Polymox_b,LFVD_SLAGMAG65B_b,LFVD_SiMn_b,LFVD_SintŽlindra_b,LFVD_SŽica_b,Last_EOP_b,PV_31CRV3_b,PV_CRMO_OSTRUŽKI_b,PV_CRV_b,PV_E3_b,PV_E6_b,PV_E8_b,PV_EMCR_b,PV_OCR12_b,PV_OCR12SP_b,PV_OCR12VM_b,PV_OCR12VM_OSTRUŽKI_b,PV_OH252_b,PV_OH255_b,PV_PK2_b,PV_PK5_b,PV_PK5M_b,PV_PT181_b,PV_UTOPMO2_b,PV_UTOPMO4_b,PV_VCMO230_b,PouringScrap,OtherScrap,TotalIngotsWeight,Cr_Last_EOP,Cr_LowerLimit,Cr_UpperLimit,Cr_Target,Ni_Last_EOP,Ni_LowerLimit,Ni_UpperLimit,Ni_Target,Mo_Last_EOP,Mo_LowerLimit,Mo_UpperLimit,Mo_Target,V_Last_EOP,V_LowerLimit,V_UpperLimit,V_Target,W_Last_EOP,W_LowerLimit,W_UpperLimit,W_Target,LFVD_AlBloki,LFVD_AlGran,LFVD_AlOplašèenaŽica,LFVD_AlŽica,LFVD_BelaŽlindra,LFVD_Boksit,LFVD_CASIfi13,LFVD_CaO,LFVD_Cfi13,LFVD_EPŽŽlindra,LFVD_FeAl,LFVD_FeCrA,LFVD_FeCrC,LFVD_FeCrC51,LFVD_FeCrCSi,LFVD_FeMnC,LFVD_FeMo,LFVD_FeS,LFVD_FeSi,LFVD_FeV,LFVD_FeW72,LFVD_KalcijevKarbid,LFVD_KarboritMleti,LFVD_MnMet,LFVD_NiGran,LFVD_Polymox,LFVD_SLAGMAG65B,LFVD_SiMn,LFVD_SintŽlindra,LFVD_SŽica,Last_EOP,PV_31CRV3,PV_CRMO_OSTRUŽKI,PV_CRV,PV_E3,PV_E6,PV_E8,PV_EMCR,PV_OCR12,PV_OCR12SP,PV_OCR12VM,PV_OCR12VM_OSTRUŽKI,PV_OH252,PV_OH255,PV_PK2,PV_PK5,PV_PK5M,PV_PT181,PV_UTOPMO2,PV_UTOPMO4,PV_VCMO230
0,0,300.0,200.0,32300.0,10.2019,11.1,11.75,11.46,0.1,0.0,0.3,0.0,0.8,0.7,0.85,0.73,0.73,0.72,0.85,0.76,0.4,0.0,0.2,0.0,75290.0,20.0,0.0,3.0,0.0,0.0,70.0,16.5,400.0,17.2,0.0,0.0,0.0,1500.0,0.0,0.0,30.0,235.0,0.0,120.0,630.0,0.0,0.0,135.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,26400.0,0.0,0.0,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,300.0,200.0,32300.0,10.2019,11.1,11.75,11.46,0.1,0.0,0.3,0.0,0.8,0.7,0.85,0.73,0.73,0.72,0.85,0.76,0.4,0.0,0.2,0.0,20.0,0.0,3.0,0.0,0.0,70.0,16.5,400.0,17.2,0.0,0.0,0.0,1500.0,0.0,0.0,30.0,235.0,0.0,120.0,630.0,0.0,0.0,135.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,26400.0,0.0,0.0,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,300.0,200.0,32300.0,10.2019,11.1,11.75,11.46,0.1,0.0,0.3,0.0,0.8,0.7,0.85,0.73,0.73,0.72,0.85,0.76,0.4,0.0,0.2,0.0,20.0,0.0,3.0,0.0,0.0,70.0,16.5,400.0,17.2,0.0,0.0,0.0,1500.0,0.0,0.0,30.0,235.0,0.0,120.0,630.0,0.0,0.0,135.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,26400.0,0.0,0.0,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
